---
title: Joint probability inference
---


In [ ]:

import msprime
import pandas as pd
import numpy as np
import sgkit as sg
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'retina'



In [ ]:
path = "/faststorage/project/baboondiversity/data/PG_panu3_phased_chromosomes_4_7_2021_ZARR/chr20.phased.rehead.vcz"

ds = sg.load_dataset(path)
keep = ['PD_0199', 'PD_0200', 'PD_0201', 'PD_0202', 'PD_0203']
mask = ds["sample_id"].isin(keep).values
ds_sub = ds.isel(samples=mask)
nr_samples = ds_sub.dims["samples"]

print("Using nr_samples =", nr_samples)


def derived_counts(ts, rec_rate):
    records = []
    for var in ts.variants():
        p, g = var.site.position, var.genotypes
        records.append((int(p), p * rec_rate, g.sum()))
    df = pd.DataFrame.from_records(
        records,columns=["pos", "gen_pos", "count"]
    )
    return df
path = "/faststorage/project/baboondiversity/data/PG_panu3_phased_chromosomes_4_7_2021_ZARR/chr20.phased.rehead.vcz"

ds = sg.load_dataset(path)

# vælg samme samples som du bruger i analyse
keep = ['PD_0199', 'PD_0200', 'PD_0201', 'PD_0202', 'PD_0203']

mask = ds["sample_id"].isin(keep).values
ds_sub = ds.isel(samples=mask)

# number of haploid samples
nr_samples = ds_sub.dims["samples"]

print("Using nr_samples =", nr_samples)



def derived_counts(ts, rec_rate):
    records = []
    
    for var in ts.variants():
        p, g = var.site.position, var.genotypes
        records.append((int(p), p * rec_rate, g.sum()))
    
    df = pd.DataFrame.from_records(
        records,
        columns=["pos", "gen_pos", "count"]
    )
    
    return df

mut_rate = 5e-10        
rec_rate = 1e-8
seq_length = 100_000_000 
pop1_size, pop2_size, anc_pop_size = 20_000, 10_000, 15_000
migr_pop1_to_pop2 = 1e-4
migr_pop2_to_pop1 = 5e-4


demography = msprime.Demography()
demography.add_population(name="pop1", initial_size=pop1_size)
demography.add_population(name="pop2", initial_size=pop2_size)
demography.set_migration_rate(source="pop1", dest="pop2", rate=migr_pop1_to_pop2)
demography.set_migration_rate(source="pop2", dest="pop1", rate=migr_pop2_to_pop1)
ts = msprime.sim_ancestry(samples={"pop1": nr_samples}, ploidy=1, 
                          demography=demography, recombination_rate=rec_rate, 
                          sequence_length=seq_length, random_seed=12)
ts = msprime.sim_mutations(ts, rate=mut_rate, random_seed=5678)
df = derived_counts(ts, rec_rate)
df.to_csv("../data/msprime_derived_counst.csv", index=False)


In [ ]:
def pairs_in_range(nums, diff_lo, diff_hi):
    n = len(nums)
    lo, hi = 1, 1
    pairs = []

    for i in range(n):
        if lo <= i:
            lo = i + 1

        while lo < n and nums[lo] - nums[i] < diff_lo:
            lo += 1

        if hi <= i:
            hi = i + 1

        while hi < n and nums[hi] - nums[i] <= diff_hi:
            hi += 1

        for j in range(lo, hi):
            pairs.append((i, j))

    return pairs
col = "possition" 
distance, tolerance = 5000, 500
min_dist, max_dist = distance - tolerance, distance + tolerance
records = []
for i, j in pairs_in_range(df[col].values, min_dist, max_dist):
    records.append((df.at[i, col], df.at[j, col], df.at[i, "derived_count"], df.at[j, "derived_count"]))
pairs = pd.DataFrame.from_records(records, columns=["pos1", "pos2", "count1", "count2"])
pairs.head()


In [ ]:

mask = (pairs.pos1 == pairs.pos1.shift()) | (pairs.pos2 == pairs.pos2.shift())
filtered_pairs = pairs.loc[~mask, :]
filtered_pairs.head()


In [ ]:

plt.hist(filtered_pairs.pos2 - filtered_pairs.pos1, bins=20)
plt.xlabel("Distance between SNPs")
plt.ylabel("Frequency")
plt.title("Distance distribution")
plt.show()


In [ ]:
nr_samples = int(df["nr_samples"].max())

n = len(filtered_pairs)
observations = np.zeros((n, nr_samples), dtype=int)
for i, pair in enumerate(filtered_pairs[["count1", "count2"]].values):
   observations[i, pair] = 1


In [ ]:
msg = f"""
Two-locus observations from baboon data:
    Number of samples: {nr_samples}
    Number of SNP pairs: {n}
    Distance window: {distance} ± {tolerance} bp
"""

print(msg)